In [1]:
import os
label_dir = r'C:\Users\DELL\OneDrive\Desktop\Oil_Spill\oil-spill\train\labels'
print("Folder exists:", os.path.exists(label_dir))
files = os.listdir(label_dir)
print("Number of files:", len(files))
print("First 5 files:", files[:5])

Folder exists: True
Number of files: 1002
First 5 files: ['img_0001.png', 'img_0002.png', 'img_0003.png', 'img_0004.png', 'img_0005.png']


In [2]:
from PIL import Image
import numpy as np
import os

label_dir = r'C:\Users\DELL\OneDrive\Desktop\Oil_Spill\oil-spill\train\labels'
all_colors = set()
count = 0

for file in os.listdir(label_dir):
    if not file.endswith(('.png', '.jpg', '.jpeg')):
        continue
    full_path = os.path.join(label_dir, file)
    img = np.array(Image.open(full_path).convert('RGB')).astype(int)
    pixels = img.reshape(-1, 3)
    unique = set(map(tuple, pixels))
    all_colors.update(unique)
    count += 1

print(f"Total files processed: {count}")

print(f"Total unique colors: {len(all_colors)}")
for color in sorted(all_colors):
    print(f"  RGB{color}")

Total files processed: 1002
Total unique colors: 5
  RGB(0, 0, 0)
  RGB(0, 153, 0)
  RGB(0, 255, 255)
  RGB(153, 76, 0)
  RGB(255, 0, 0)


In [4]:
import cv2
import numpy as np
import os

label_dir  = r'C:\Users\DELL\OneDrive\Desktop\Oil_Spill\oil-spill\train\labels'
output_dir = r'C:\Users\DELL\OneDrive\Desktop\Oil_Spill\oil-spill\train\binary_mask'
os.makedirs(output_dir, exist_ok=True)

for file in os.listdir(label_dir):
    if not file.endswith('.png'):
        continue

    img = cv2.imread(os.path.join(label_dir, file))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    lower_oil = np.array([0,   200, 200])
    upper_oil = np.array([50,  255, 255])
    oil_mask  = cv2.inRange(img, lower_oil, upper_oil)

    binary_label = (oil_mask / 255).astype(np.uint8)
    visual_mask = binary_label * 255

    cv2.imwrite(os.path.join(output_dir, file), visual_mask)

count = len(os.listdir(output_dir))
print(f" {count} masks saved")


 1002 masks saved


In [5]:

mask_dir = r'C:\Users\DELL\OneDrive\Desktop\Oil_Spill\oil-spill\train\binary_mask'

total_files = 0
has_oil = 0
no_oil = 0
corrupt = 0

for file in os.listdir(mask_dir):
    if not file.endswith('.png'):
        continue
    
    total_files += 1
    mask = cv2.imread(os.path.join(mask_dir, file), 0)
    
    if mask is None:
        corrupt += 1
        continue
    
    if np.any(mask == 255):
        has_oil += 1
    else:
        no_oil += 1

print(f"Total masks     : {total_files}")
print(f"Has oil (white) : {has_oil}")
print(f"No oil          : {no_oil}")
print(f"Corrupt files   : {corrupt}")
print(f"Oil %           : {has_oil/total_files*100:.1f}%")



Total masks     : 1002
Has oil (white) : 791
No oil          : 211
Corrupt files   : 0
Oil %           : 78.9%
